# Chatbot QA Notebook

Run cells from top to bottom.

## Step 0: Imports and Environment
Load dependencies and optional `.env` variables.

In [ ]:
import os

# Step 0: Load environment variables from .env when available.
# This allows secrets (like HF_TOKEN) to be configured without hardcoding.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# PromptTemplate formats context+question before sending to the LLM.
from langchain_core.prompts import PromptTemplate
# Embedding model turns text chunks and user queries into vectors.
from langchain_huggingface import HuggingFaceEmbeddings
# FAISS stores vectors and performs similarity search over chunks.
from langchain_community.vectorstores import FAISS
# HuggingFacePipeline wraps a transformers pipeline for LangChain.
from langchain_community.llms import HuggingFacePipeline
# RetrievalQA combines retriever + prompt + LLM into one callable chain.
from langchain_classic.chains import RetrievalQA
# Transformers objects used to load local model/tokenizer and generate text.
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

d:\Work\Chatbot-file-tuning-alzheimer-diesease\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Paths and Prompt Template
Define constants, local model path helper, and answer style template.

In [ ]:
# Folder where ingest.ipynb saved the FAISS index.
DB_FAISS_PATH = 'vectorstore/db_faiss'
# Optional token for gated/private Hugging Face downloads.
HF_TOKEN = os.getenv('HF_TOKEN')

# In notebooks, __file__ is usually undefined.
# Use current working directory so relative paths still work in Jupyter.
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()


def _default_local_llm_path(model_id: str) -> str:
    # Map a model id like owner/model into a safe local folder name.
    safe_model_name = model_id.replace('/', '__')
    return os.path.join(BASE_DIR, 'local_models', safe_model_name)


custom_prompt_template = """
Answer the question based only on the following context:
{context}
You are allowed to rephrase the answer based on the context.
Keep the response concise (2-4 sentences).
Question: {question}
Only return the helpful answer below and nothing else.
Helpful answer:
"""

## Step 2: Prompt and Retrieval Functions
Build the prompt object and retrieval QA chain.

In [ ]:
# Step 1: Create the prompt template that controls answer style and length.
def set_custom_prompt():
    """Build the final prompt object used in RetrievalQA."""
    # The chain will inject retrieved chunk text into {context} and user input into {question}.
    prompt = PromptTemplate(template=custom_prompt_template,
                            input_variables=['context', 'question'])
    return prompt


# Retrieval QA Chain
def retrieval_qa_chain(llm, prompt, db):
    # Retrieve top matching chunks from FAISS, then generate an answer from them.
    # k=2 means the retriever returns the 2 most relevant chunks per question.
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type='stuff',
        # 'stuff' places all retrieved chunks into a single prompt context block.
        retriever=db.as_retriever(search_kwargs={'k': 2}),
        # Include source docs so we can print citations in the notebook response.
        return_source_documents=True,
        chain_type_kwargs={'prompt': prompt}
    )
    return qa_chain

## Step 3: Load LLM
Load from local cache when available, otherwise download and cache.

In [ ]:
# Loading the model
def load_llm():
    # Step 2: Load or download the language model used to generate answers.
    # Prefer a previously downloaded local model to avoid repeated downloads.
    model_id = os.getenv('LLM_MODEL_ID', 'HuggingFaceTB/SmolLM2-360M-Instruct')
    local_llm_path = os.getenv('LOCAL_LLM_PATH', _default_local_llm_path(model_id))

    # A model folder is considered ready when key config exists.
    local_model_ready = os.path.isdir(local_llm_path) and os.path.exists(
        os.path.join(local_llm_path, 'config.json')
    )

    if local_model_ready:
        # Fast path: load model files from disk only (offline-friendly).
        tokenizer = AutoTokenizer.from_pretrained(local_llm_path, local_files_only=True)
        model = AutoModelForCausalLM.from_pretrained(local_llm_path, local_files_only=True)
    else:
        # Fallback path: download once, then cache under local_models/ for next runs.
        tokenizer_kwargs = {}
        model_kwargs = {}
        if HF_TOKEN:
            tokenizer_kwargs['token'] = HF_TOKEN
            model_kwargs['token'] = HF_TOKEN

        tokenizer = AutoTokenizer.from_pretrained(model_id, **tokenizer_kwargs)
        model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)

        # Save downloaded model files so future runs are offline and faster.
        os.makedirs(local_llm_path, exist_ok=True)
        tokenizer.save_pretrained(local_llm_path)
        model.save_pretrained(local_llm_path)

    # Wrap model+tokenizer into a text-generation pipeline LangChain can call.
    # do_sample=False makes output deterministic and usually more stable for QA.
    text_generation_pipeline = pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=120,
        do_sample=False,
        return_full_text=False
    )

    llm = HuggingFacePipeline(pipeline=text_generation_pipeline)
    return llm

## Step 4: Build QA Bot and Query Helper
Create the end-to-end QA pipeline and a helper for single questions.

In [ ]:
# QA Model Function
def qa_bot():
    # Step 3: Build the full QA pipeline (embeddings + retriever + LLM + prompt).
    embedding_kwargs = {'device': 'cpu'}
    if HF_TOKEN:
        embedding_kwargs['token'] = HF_TOKEN

    # This embedding model must match ingest.ipynb so query/document vectors live in same space.
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2',
                                       model_kwargs=embedding_kwargs)
    # Load the existing vector index built by ingest.py / ingest.ipynb.
    db = FAISS.load_local(DB_FAISS_PATH, embeddings, allow_dangerous_deserialization=True)

    # Load the language model used to compose final answers from retrieved context.
    llm = load_llm()

    # Build the prompt template that controls response style and constraints.
    qa_prompt = set_custom_prompt()

    # Assemble retriever + prompt + llm into a callable chain.
    qa = retrieval_qa_chain(llm, qa_prompt, db)

    return qa


# Output function
def final_result(query):
    # Convenience helper for one-off questions outside the interactive cell.
    # Returns the raw dict (answer + source_documents) from RetrievalQA.
    qa_result = qa_bot()
    response = qa_result({'query': query})
    return response

## Step 5: Notebook Chat (Edit-and-Run)
Notebook kernels may block `input()` prompts. Type your question by editing `user_query` in the next cell, then run that cell each time.

In [ ]:
# Notebook chat interface

# Build the chain once and reuse it across runs.
# Reusing avoids reloading model/vector store every time you ask a question.
if 'chain' not in globals():
    chain = qa_bot()

# Type your question inside the quotes, then run this cell.
user_query = "What are early symptoms of Alzheimer's disease?"

if not user_query.strip():
    print("Please set user_query to a question and run again.")
else:
    # Invoke retrieval + generation for the user query.
    res = chain.invoke({'query': user_query})
    answer = res.get('result', '')
    sources = res.get('source_documents', [])

    if sources:
        citations = []
        # Show up to 3 unique source references to keep output short and useful.
        for doc in sources[:3]:
            metadata = getattr(doc, 'metadata', {}) or {}
            source_name = os.path.basename(metadata.get('source', 'unknown'))
            page = metadata.get('page')
            if page is not None:
                citations.append(f"{source_name} (p.{page})")
            else:
                citations.append(source_name)

        unique_citations = list(dict.fromkeys(citations))
        answer += '\nSources: ' + ', '.join(unique_citations)
    else:
        answer += '\nNo sources found'

    print(f"You: {user_query}\n")
    print(f"Bot: {answer}\n")

You: What are early symptoms of Alzheimer's disease?

Bot: Early symptoms of Alzheimer's disease include memory loss, confusion, and disorientation. People with Alzheimer's may also experience difficulty with language, such as trouble finding the right words or understanding what others are saying. They may also have trouble with problem-solving, learning new things, and remembering recent events.

Answer the question based only on the following context:
Alzheimer’s Disease: Unraveling the Mystery 26
Moderate AD
By this stage, AD damage has spread further to
the areas of the cerebral cortex that control
language, reasoning, sensory processing, and
Sources: 2024 ALZHEIMER’S DISEASE FACTS AND FIGURES.pdf (p.177), 2024 ALZHEIMER’S DISEASE FACTS AND FIGURES.pdf (p.174)

